# Stage 1 — Danish taxon list

Builds the candidate species list for Denmark from GBIF: occurrence facets filtered to
`country=DK`, cross-referenced against the GBIF Backbone for accepted names, full rank
lineage, and synonyms.

Output: `taxa_raw.json` in the Drive cache.

**No GPU needed** — set the runtime to CPU and save your T4 quota for stage 3.

Runtime is 30-90 minutes depending on `--max-taxa`, almost all of it waiting politely on
a free public API. It caches to Drive, so a disconnect costs this stage and nothing else.


In [ ]:
#@title Mount Drive and install the package
from google.colab import drive
drive.mount('/content/drive')

CACHE = '/content/drive/MyDrive/life-list/cache'  #@param {type:"string"}
REPO  = '/content/life-list'                       #@param {type:"string"}

import os
os.makedirs(CACHE, exist_ok=True)

if not os.path.exists(REPO):
    !git clone https://github.com/StefanWiswedel/Life-List.git {REPO}
else:
    !cd {REPO} && git pull --ff-only

!pip -q install -e {REPO}/training
print('cache:', CACHE)


## Build the list

`--min-occurrences` is the first coarse filter. A species with three national records
will not have 80 usable photos either, so excluding it here saves a great many requests.


In [ ]:
!lifelist-taxa \
    --country DK \
    --max-taxa 20000 \
    --min-occurrences 10 \
    --cache-dir {CACHE} \
    --verbose


## What came back

In [ ]:
import json
from collections import Counter

with open(f'{CACHE}/taxa_raw.json') as fh:
    data = json.load(fh)

taxa = data['taxa']
print(f"{len(taxa)} taxa, {len(data['synonyms'])} synonyms\n")

for kingdom, count in Counter(
    t['lineage_names'].get('kingdom', '(unplaced)') for t in taxa
).most_common():
    print(f'  {kingdom:20} {count:6}')

missing = sum(1 for t in taxa if not t['vernacular_en'])
print(f'\n{missing} taxa without an English vernacular name')


---

Next: **stage 2**, which filters the iNaturalist open data and prints the taxon-count
report. That report is a decision point — the pipeline stops there on purpose.
